# VERSION A - Python + OpenAI 

## Orchestration

This version uses only the OpenAI Python SDK without LangChain or LangGraph.

The agent orchestration is handled manually

# Imports and OpenAI Setup

In [2]:
import os, json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY not found. "
        "Check your .env file and restart the kernel."
    )

client = OpenAI(api_key=API_KEY)

MODEL = "gpt-4o-mini"

print("Connected. Model:", MODEL)

Connected. Model: gpt-4o-mini


# Tool Functions

In [ ]:
City_Costs = {
    "ooty": {
        "budget": 2500,
        "mid-range":4500,
        "luxury": 8000
    },
    "manali": {
        "budget": 3000,
        "mid-range": 5500,
        "luxury": 10000
    },
    "bangkok": {
        "budget": 3500,
        "mid-range": 6500,
        "luxury": 12000
    },
    "bali": {
        "budget": 4000,
        "mid-range": 7500,
        "luxury": 14000
    }
}

Default_Costs = {
    "budget": 3000,
    "mid-range": 5500,
    "luxury": 10000
}

def get_destination_weather(city: str, days_ahead: int = 7) -> dict:
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params = {
            "name": city,
            "count": 1
        },
        timeout=10
    )

    geo_response.raise_for_status()
    geo_data = geo_response.json()
    if not geo_data.get("results"):
        raise ValueError(
            f"Could not find city: {city}"
        )
    place = geo_data["results"][0]
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": (
                "temperature_2m_max,"
                "temperature_2m_min,"
                "precipitation_probability_max"
            ),
            "timezone": "auto",
            "forecast_days": 16
        },
        timeout=10
    )

    weather_response.raise_for_status()
    data = weather_response.json()["daily"]

    if days_ahead >= len(data["time"]):
        raise ValueError(
            f"Weather forecast unavailable {days_ahead} days ahead."
        )
    return {
        "city": place["name"],
        "date": data["time"][days_ahead],
        "min_temp_c": data["temperature_2m_min"][days_ahead],
        "max_temp_c": data["temperature_2m_max"][days_ahead],
        "rain_probability": data["precipitation_probability_max"][days_ahead]
    }


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    response = requests.get(
        "https://api.frankfurter.app/latest",
        params={
            "amount": amount,
            "from": from_currency,
            "to": to_currency
        },
        timeout=10
    )

    response.raise_for_status()
    data = response.json()

    if to_currency not in data.get("rates", {}):
        raise ValueError(
            f"Could not convert {from_currency} to {to_currency}"
        )
    converted_amount = data["rates"][to_currency]

    return {
        "amount": amount,
        "from_currency": from_currency,
        "to_currency": to_currency,
        "converted_amount": round(converted_amount, 2)
    }

def estimate_trip_cost(city: str, days: int, travel_style: str = "budget") -> dict:
    city_key = city.lower()

    if city_key in City_Costs:
        cost_per_day = City_Costs[city_key][travel_style]
        return {
            "city": city,
            "days": days,
            "travel_style": travel_style,
            "estimated_total_cost": cost_per_day * days
        }
    cost_per_day = Default_Costs[travel_style]

    return {
        "city": city,
        "days": days,
        "travel_style": travel_style,
        "currency": "INR",
        "estimated_total_cost": cost_per_day * days,
        "note": "Unknown city. Default cost tier applied."
    }
    


# Manual Tool Schemas

In [16]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_destination_weather",
        "description": (
            "Get the weather forecast for a destination city. "
            "Use this for weatheer, temperature, rain, "
            "or packing questions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Destination city."
                },
                "days_ahead": {
                    "type": "integer",
                    "description": (
                        "Number of days ahead. "
                        "Use 1 for tomorrow and 7 for next week."
                    )
                }
            },
            "required": ["city"]
        }
    }
}


currency_tool = {
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Convert money between currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number"
                },
                "from_currency": {
                    "type": "string"
                },
                "to_currency": {
                    "type": "string"
                }
            },
            "required": [
                "amount",
                "from_currency",
                "to_currency"
            ]
        }
    }
}


cost_tool = {
    "type": "function",
    "function": {
        "name": "estimate_trip_cost",
        "description": (
            "Estimate trip cost in INR "
            "based on city, number of days, "
            "and travel style."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                },
                "days": {
                    "type": "integer"
                },
                "travel_style": {
                    "type": "string",
                    "enum": ["budget", "mid-range", "luxury"]
                }
            },
            "required": ["city", "days"]
        }
    }
}

TOOL_SCHEMAS = [weather_tool, currency_tool, cost_tool]

# Tool Registry

In [17]:
Tools = {
    "get_destination_weather": get_destination_weather,
    "convert_currency": convert_currency,
    "estimate_trip_cost": estimate_trip_cost
}

print(Tools)

{'get_destination_weather': <function get_destination_weather at 0x0000022C046940E0>, 'convert_currency': <function convert_currency at 0x0000022C046949A0>, 'estimate_trip_cost': <function estimate_trip_cost at 0x0000022C0800A980>}


In [18]:
Agent_system = (
    "You are TripBuddy, a travel planning assistant."
    "Use the available tools when needed."
    "You can call multiple tools."
    "Use previous conversation information for follow-up questions."
    "If a tool returns an error, explain the problem clearly."
)

memory = {}

# Final Agent Loop

In [19]:
def run_tripbuddy(user_message, session_id="default", max_turns=6, verbose=True, tools=None, schemas=None):
    tools = tools or Tools
    schemas = schemas or TOOL_SCHEMAS

    if session_id not in memory:
        memory[session_id] = [
            {
                "role": "system",
                "content": Agent_system
            }
        ]

    messages = memory[session_id]

    messages.append({
        "role": "user",
        "content": user_message
    })


    if verbose:
        print(f"USER: {user_message}\n")

    turn = 0

    while True:

        turn += 1

        if turn > max_turns:
            return (
                f"Stopped after {max_turns} turns "
                "without a final answer."
            )


        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=schemas,
            temperature=0
        )

        message = response.choices[0].message


        if not message.tool_calls:

            messages.append(
                message.model_dump(
                    exclude_none=True
                )
            )

            if verbose:
                print(f"TURN {turn} - final answer:\n")

            return message.content


        messages.append(
            message.model_dump(
                exclude_none=True
            )
        )


        for call in message.tool_calls:

            name = call.function.name

            try:
                args = json.loads(call.function.arguments)

                if name not in tools:
                    raise KeyError(
                        f"No such tool: {name}"
                    )

                result = tools[name](**args)


            except Exception as e:

                result = {
                    "error": f"{type(e).__name__}: {e}"
                }

                args = call.function.arguments


            if verbose:

                print(
                    f"TURN {turn} - "
                    f"{name}({args})"
                )

                print(
                    f"{json.dumps(result)}\n"
                )


            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })


## Query 1 — Weather in Ooty

What's the weather in Ooty tomorrow?

In [20]:
run_tripbuddy(
    "What's the weather in Ooty tomorrow?",
    session_id="query1",
    verbose=True
)

USER: What's the weather in Ooty tomorrow?

TURN 1 - get_destination_weather({'city': 'Ooty', 'days_ahead': 1})
{"city": "Udhagamandalam", "date": "2026-09-13", "min_temp_c": 14.1, "max_temp_c": 20.0, "rain_probability": 47}

TURN 2 - final answer:



"Tomorrow in Ooty (Udhagamandalam), the weather is expected to have a minimum temperature of 14.1°C and a maximum temperature of 20.0°C. There's a 47% chance of rain, so it might be a good idea to carry an umbrella or raincoat if you're heading out!"

## Query 2 — Weather and Trip Cost for Manali

I'm going to Manali for 4 days next week — should I pack for rain, and roughly what will it cost?

In [21]:
run_tripbuddy(
    "I'm going to Manali for 4 days next week — should I pack for rain, and roughly what will it cost?",
    session_id="manali_trip",
    verbose=True
)

USER: I'm going to Manali for 4 days next week — should I pack for rain, and roughly what will it cost?

TURN 1 - get_destination_weather({'city': 'Manali', 'days_ahead': 7})
{"city": "Manali", "date": "2026-09-19", "min_temp_c": 24.2, "max_temp_c": 33.1, "rain_probability": 89}

TURN 1 - estimate_trip_cost({'city': 'Manali', 'days': 4})
{"city": "Manali", "days": 4, "travel_style": "budget", "estimated_total_cost": 12000}

TURN 2 - final answer:



"Next week in Manali, the weather forecast indicates a high probability of rain (89%), with temperatures ranging from 24.2°C to 33.1°C. It's advisable to pack for rain, including waterproof clothing and accessories.\n\nAs for the cost, a budget trip to Manali for 4 days is estimated to be around ₹12,000."

## Query 3 Currency Conversion

Convert 500 USD to INR for my Bali trip.

In [22]:
run_tripbuddy(
    "Convert 500 USD to INR for my Bali trip.",
    session_id="query3",
    verbose=True
)

USER: Convert 500 USD to INR for my Bali trip.

TURN 1 - convert_currency({'amount': 500, 'from_currency': 'USD', 'to_currency': 'INR'})
{"amount": 500, "from_currency": "USD", "to_currency": "INR", "converted_amount": 47778}

TURN 2 - final answer:



'500 USD is approximately 47,778 INR for your Bali trip. If you need any more assistance with your travel plans, feel free to ask!'

## Query 4 — Bangkok Trip Planning

Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?

In [23]:
run_tripbuddy(
    "Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?",
    session_id="bangkok_trip",
    verbose=True
)

USER: Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?

TURN 1 - get_destination_weather({'city': 'Bangkok', 'days_ahead': 1})
{"city": "Bangkok", "date": "2026-09-13", "min_temp_c": 24.2, "max_temp_c": 30.6, "rain_probability": 62}

TURN 1 - estimate_trip_cost({'city': 'Bangkok', 'days': 6})
{"city": "Bangkok", "days": 6, "travel_style": "budget", "currency": "INR", "estimated_total_cost": 18000, "note": "Unknown city. Default cost tier applied."}

TURN 1 - convert_currency({'amount': 20000, 'from_currency': 'THB', 'to_currency': 'INR'})
{"amount": 20000, "from_currency": "THB", "to_currency": "INR", "converted_amount": 57798}

TURN 2 - final answer:



"Here's the information for your 6-day trip to Bangkok:\n\n### Weather\n- **Date**: September 13, 2026\n- **Min Temperature**: 24.2°C\n- **Max Temperature**: 30.6°C\n- **Rain Probability**: 62%\n\n### Estimated Trip Cost\n- **Total Cost**: ₹18,000 (for a budget travel style)\n\n### Currency Conversion\n- **20,000 THB** is approximately **₹57,798**.\n\nIf you have any more questions or need further assistance, feel free to ask!"

## Query 5 — Invalid City Error Handling

What's the weather in Zzyxville next week?

In [24]:
run_tripbuddy(
    "What's the weather in Zzyxville next week?",
    session_id="query5",
    verbose=True
)

USER: What's the weather in Zzyxville next week?

TURN 1 - get_destination_weather({"city":"Zzyxville","days_ahead":7})
{"error": "ValueError: Could not find city: Zzyxville"}

TURN 2 - final answer:



'It seems that "Zzyxville" is not a recognized city. Could you please check the spelling or provide the name of a different city?'

## Query 6 — Follow-up Using Memory

Make that a mid-range trip instead.

In [25]:
run_tripbuddy(
    "Make that a mid-range trip instead.",
    session_id="manali_trip",
    verbose=True
)

USER: Make that a mid-range trip instead.

TURN 1 - estimate_trip_cost({'city': 'Manali', 'days': 4, 'travel_style': 'mid-range'})
{"city": "Manali", "days": 4, "travel_style": "mid-range", "estimated_total_cost": 22000}

TURN 2 - final answer:



'For a mid-range trip to Manali for 4 days, the estimated total cost would be around ₹22,000.'